# 08b The paper's image features for THINGS-EEG1

**Kaggle:** GPU T4, Internet on. **Inputs:** the outputs of 03b (for the equality check) and 08a (the image lists).

The EEG analyses use the THINGS-EEG2 paper's features: AlexNet's feature maps, standardized, then a kernel PCA to
1,000 components, all fitted on THINGS-EEG2's 16,540 training images (03b). This notebook refits that feature space
exactly as 03b did and transforms THINGS-EEG1's images into it. Images THINGS-EEG2 also has are read from its
copies, the rest from the THINGS originals (`images_THINGS.zip` on OSF, research use only; only the features leave
the session), and a check measures how much the image source changes the features on 200 images that exist in
both.

In [ ]:
import os
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run 08b on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
import hashlib
import resource
import shutil
import time

import numpy as np
import torch

from cross_subject_encoding.data import things_eeg1 as e1
from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.data import things_images as ti
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.preflight import preflight
from cross_subject_encoding.utils.selfcheck import check, headline
from cross_subject_encoding.utils.storage import storage_plan

RUN_FULL_FEATURES = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PRETRAINED = True
WORKERS = 2
N_SOURCE_CHECK = 200  # images read from both sources
KERNEL_ROWS = 128
KERNEL_FEATURES = 4096
TMP = Path("/kaggle/tmp") if IS_KAGGLE else C.PROJECT_ROOT / "data" / "raw"
T0 = time.time()
minutes = lambda: f"{(time.time() - T0) / 60:.1f} min"
if not IS_KAGGLE:
    raise RuntimeError("08b real-data readiness/extraction runs on Kaggle")
info = preflight("cross-subject-brain-encoding-08b-outputs", inputs=["paper_alexnet_kpca1000.npz", "eeg1_image_metadata.npy"], need_gpu=RUN_FULL_FEATURES, min_gpu_gb=4, expect_output_gb=0.3)
report_path = C.METRICS_DIR / "08b_readiness.json"
FEATURES_VERIFIED = False
metadata_path = C.find_artifact("eeg1_image_metadata.npy")
reference_path = C.find_artifact("paper_alexnet_kpca1000.npz")
metadata = np.load(metadata_path, allow_pickle=True).item()
assert len(metadata["train_img_files"]) == 22248
assert len(metadata["test_img_files"]) == 200
with np.load(reference_path) as reference:
    assert reference["train"].shape == (16540, 1000)
scratch_required = 22648 * pl.N_FEATURES * 2 + 16540**2 * 8
storage = storage_plan({
    "feature_kernel_scratch": (TMP / "08b_scratch", scratch_required),
    "image_archive_extraction_reserve": (TMP, 8 * 2**30),
    "working_output_reserve": (C.DATA_PROCESSED, 2**30),
    "export_staging_reserve": (Path("/tmp/kaggle-export"), 2**30),
    "weight_cache_reserve": (Path(torch.hub.get_dir()), 2**29),
})
disk_free = storage["locations"]["feature_kernel_scratch"]["free_bytes"]
report = {"mode": "full" if RUN_FULL_FEATURES else "readiness_only",
          "metadata": str(metadata_path), "reference": str(reference_path),
          "scratch_bytes_estimate": scratch_required, "free_disk_bytes": disk_free,
          "free_disk_path": storage["locations"]["feature_kernel_scratch"]["path"],
          "storage": storage,
          "features_verified": False, "image_access_tested": False,
          "note": "Readiness does not measure full-scale resource peaks or feature fidelity."}
report_path.write_text(json.dumps(report, indent=2))
if IS_KAGGLE:
    memory = dict(line.split(":", 1) for line in Path("/proc/meminfo").read_text().splitlines())
    available = int(memory["MemAvailable"].split()[0]) * 1024
    for limit_path, used_path in [("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
                                  ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes")]:
        if Path(limit_path).exists() and Path(used_path).exists():
            limit = Path(limit_path).read_text().strip()
            if limit != "max":
                available = min(available, int(limit) - int(Path(used_path).read_text()))
    print(f"available host headroom: {available / 2**30:.1f} GiB")
    report["available_host_bytes"] = available
    report_path.write_text(json.dumps(report, indent=2))
    if RUN_FULL_FEATURES and available < 8 * 2**30:
        raise RuntimeError("08b needs at least 8 GiB of available host headroom for the dense KPCA fit")
if RUN_FULL_FEATURES and not storage["sufficient"]:
    raise RuntimeError("Insufficient combined storage reserve on an actual destination device; inspect 08b_readiness.json")
print(json.dumps(report, indent=2))

## The images

THINGS-EEG2's own image set first (as 03b), then the THINGS images it
lacks, extracted from the password-protected zip.

In [ ]:
if RUN_FULL_FEATURES:
    search = Path("/kaggle/input") if IS_KAGGLE else C.PROJECT_ROOT / "data" / "raw" / "things_images"
    roots = {sp: ti.find_image_root(search, sp) for sp in ("training", "test")}
    if not all(roots.values()):
        roots = ti.download_images(TMP / "things_images" if IS_KAGGLE else search)
    meta2 = te.load_image_metadata(C.EEG_RAW if C.EEG_RAW.is_dir() else None)
    paths2 = {sp: ti.image_paths(roots[sp], meta2, sp) for sp in ("training", "test")}
    eeg2_by_file = {p.name: p for sp in ("training", "test") for p in paths2[sp]}  # THINGS file names are unique
    eeg2 = {}
    m1 = np.load(C.find_artifact("eeg1_image_metadata.npy"), allow_pickle=True).item()
    train1 = [f"{c}/{f}" for c, f in zip(m1["train_img_concepts"], m1["train_img_files"])]
    val1 = [f"{c}/{f}" for c, f in zip(m1["test_img_concepts"], m1["test_img_files"])]
    eeg2 = {k: eeg2_by_file[k.rsplit("/", 1)[-1]] for k in train1 + val1 if k.rsplit("/", 1)[-1] in eeg2_by_file}
    need = [k for k in train1 + val1 if k not in eeg2]
    in_both = [k for k in train1 if k in eeg2]
    check_keys = list(np.random.default_rng(C.SEED).choice(in_both, N_SOURCE_CHECK, replace=False))
    print(f"EEG1: {len(train1)} training, {len(val1)} validation | from THINGS-EEG2's copies: {len(in_both)} "
          f"({sum(k in eeg2 and eeg2[k] in paths2['test'] for k in train1)} of them THINGS-EEG2 test images) | from THINGS: {len(need)} | {minutes()}")
    zip_path = TMP / "images_THINGS.zip"
    if not zip_path.exists():
        e1.fetch(f"https://osf.io/download/{e1.THINGS_ZIP_OSF}/", zip_path)
    things = e1.extract_things_images(zip_path, need + check_keys, TMP / "things_full")
    zip_path.unlink()
    print(f"extracted {len(things)} THINGS images | {minutes()}")

## Features, fitted on THINGS-EEG2's training images as in 03b

The fit set and its order are 03b's, so the training components must
equal 03b's file; every other image is only transformed.

In [ ]:
if RUN_FULL_FEATURES:
    fit_paths = paths2["training"]
    rest_keys = train1 + val1
    rest_paths = [eeg2[k] if k in eeg2 else things[k] for k in rest_keys] + [things[k] for k in check_keys]
    all_paths = fit_paths + rest_paths
    unique_paths, inverse = pl.unique_image_paths(all_paths)
    fit_rows = inverse[:len(fit_paths)]
    assert np.array_equal(fit_rows, np.arange(len(fit_paths))), "fit image order or uniqueness changed"
    query_rows = np.arange(len(fit_paths), len(unique_paths))
    scratch = TMP / "08b_scratch"
    scratch.mkdir(parents=True, exist_ok=True)
    feature_path, kernel_path = scratch / "features.npy", scratch / "fit_kernel.npy"
    scratch_bytes = len(unique_paths) * pl.N_FEATURES * 2 + len(fit_paths) ** 2 * 8
    free_bytes = shutil.disk_usage(scratch).free
    if free_bytes < scratch_bytes + 2**30:
        raise RuntimeError(f"08b scratch needs {scratch_bytes / 2**30:.1f} GiB plus 1 GiB reserve; only {free_bytes / 2**30:.1f} GiB free")
    print(f"{len(all_paths)} requested rows -> {len(unique_paths)} unique paths; scratch {scratch_bytes / 1e9:.1f} GB")
    x, _ = pl.extract(unique_paths, DEVICE, batch=64, workers=WORKERS, pretrained=PRETRAINED,
                      output_path=feature_path, progress=lambda i, n: print(f"  {i}/{n}", end="\r"))
    print(f"disk-backed features {x.shape}, {x.nbytes / 1e9:.1f} GB | {minutes()}")
    mean, scale = pl.standardize_stats_batched(x, fit_rows, feature_chunk=KERNEL_FEATURES)
    kernel_start = time.time()
    def kernel_progress(done, total):
        if done == min(KERNEL_ROWS, total):
            print(f"fit-kernel projection from first tile: {(time.time() - kernel_start) * total / done / 60:.1f} min")
    k_fit = pl.training_kernel_batched(x, mean, scale, fit_rows, DEVICE, kernel_path,
                                       row_chunk=KERNEL_ROWS, feature_chunk=KERNEL_FEATURES, progress=kernel_progress)
    n_comp = min(pl.N_COMPONENTS, len(fit_rows) - 1)
    query_batches = pl.polynomial_kernel_batches(x, mean, scale, fit_rows, query_rows, DEVICE,
                                                 row_chunk=KERNEL_ROWS, feature_chunk=KERNEL_FEATURES)
    z_fit, z_extra, kp = pl.kernel_pca_batched(k_fit, query_batches, len(query_rows), n_comp, row_chunk=KERNEL_ROWS)
    z_unique = np.concatenate([z_fit, z_extra])
    z_rest = z_unique[inverse[len(fit_paths):]]
    del kp, k_fit, x, z_unique, z_extra
    feature_path.unlink()
    kernel_path.unlink()
    n1, nv = len(train1), len(val1)
    z_train1, z_val1, z_check_things = z_rest[:n1], z_rest[n1:n1 + nv], z_rest[n1 + nv:]
    pos = {kk: i for i, kk in enumerate(train1)}
    z_check_eeg2 = z_train1[[pos[kk] for kk in check_keys]]
    print(f"kernel PCA: {z_fit.shape[1]} components | {minutes()}")
    print(f"main-process peak host RSS: {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20:.2f} GiB")
    for peak_path in ["/sys/fs/cgroup/memory.peak", "/sys/fs/cgroup/memory/memory.max_usage_in_bytes"]:
        if Path(peak_path).exists():
            print(f"cgroup peak memory: {int(Path(peak_path).read_text()) / 2**30:.2f} GiB")
            break
    if DEVICE == "cuda":
        print(f"peak GPU allocated: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB; reserved: {torch.cuda.max_memory_reserved() / 2**30:.2f} GiB")

## Checks

The refit against 03b's training components, and the image source (THINGS
original against THINGS-EEG2's copy) on the same images.

In [ ]:
if RUN_FULL_FEATURES:
    with np.load(C.find_artifact("paper_alexnet_kpca1000.npz")) as f:
        z03b = f["train"].astype(np.float64)
    comp_r = np.array([abs(np.corrcoef(z_fit[:, j], z03b[:, j])[0, 1]) for j in range(z_fit.shape[1])])
    signs = np.where(np.einsum("ij,ij->j", z_fit, z03b) < 0, -1.0, 1.0)
    z_fit = z_fit * signs
    z_train1 = z_train1 * signs
    z_val1 = z_val1 * signs
    z_check_things = z_check_things * signs
    z_check_eeg2 = z_check_eeg2 * signs
    refit_err = float(np.abs(z_fit - z03b).max() / np.abs(z03b).max())
    var = z_fit.var(0)
    w = var / var.sum()
    src_r = np.array([np.corrcoef(a * np.sqrt(w), b * np.sqrt(w))[0, 1] for a, b in zip(z_check_things, z_check_eeg2)])
    print(f"refit vs 03b: smallest |r| per component {comp_r.min():.6f} (first 100: {comp_r[:100].min():.6f}), largest relative error {refit_err:.2e}")
    print(f"THINGS original vs THINGS-EEG2 copy, variance-weighted r per image: median {np.median(src_r):.4f}, lowest {src_r.min():.4f}")

In [ ]:
if RUN_FULL_FEATURES:
    rows = [
        {"measure": "EEG1 training images with features", "measured": float(len(z_train1)), "expected": "22248", "ok": (22248, 22248), "look_again": "an image is missing"},
        {"measure": "EEG1 training images read from THINGS-EEG2's copies", "measured": float(len(in_both)), "expected": "16718 (16,540 training + 178 test)", "ok": (16718, 16718),
         "look_again": "the overlap differs from the events tables' count"},
        {"measure": "refit against 03b: largest error relative to the components' range", "measured": refit_err, "expected": "below 1e-3", "ok": (None, 1e-3),
         "look_again": "the refit is not 03b's feature space (look at the per-component |r|: a sign flip only would leave it at 1)"},
        {"measure": "refit against 03b: smallest |r| over the first 100 components", "measured": float(comp_r[:100].min()), "expected": "above 0.999", "ok": (0.999, None),
         "look_again": "the leading components moved: the replication's features are not 03b's"},
        {"measure": "image source: median variance-weighted r, THINGS original vs THINGS-EEG2 copy", "measured": float(np.median(src_r)), "expected": "0.95 or more", "ok": (0.95, None),
         "look_again": "THINGS-EEG2's copies were processed (cropped or resized) differently: read all EEG1 images from THINGS instead"},
    ]
    status = check("08b", rows, C.METRICS_DIR)
    FEATURES_VERIFIED = all(row["status"] == "OK" for row in status) and all(
        np.isfinite(array).all() for array in (z_train1, z_val1, src_r, comp_r))
    report.update(features_verified=bool(FEATURES_VERIFIED), image_access_tested=True,
                  refit_error=refit_err,
                  sign_flips=int((signs < 0).sum()), source_r_median=float(np.median(src_r)))
    if FEATURES_VERIFIED:
        output_path = C.DATA_PROCESSED / "eeg1_paper_kpca1000.npz"
        fit_names = {path.name for path in fit_paths}
        np.savez(output_path, train=z_train1.astype(np.float32), test=z_val1.astype(np.float32),
                 image=np.array(train1 + val1),
                 kpca_fit_member=np.array([key.rsplit("/", 1)[-1] in fit_names for key in train1]))
        digest = hashlib.sha256()
        with output_path.open("rb") as stream:
            for block in iter(lambda: stream.read(2**20), b""):
                digest.update(block)
        report["feature_sha256"] = digest.hexdigest()
    else:
        print("FEATURE GATE FAILED: exporting diagnostics only; do not use these features")
    report_path.write_text(json.dumps(report, indent=2))
    headline("08b", {"train": len(z_train1), "val": len(z_val1), "from EEG2 copies": len(in_both), "from THINGS": len(need), "refit error": refit_err,
                     "source r median": float(np.median(src_r))}, C.METRICS_DIR)

## Save outputs as a Kaggle Dataset

Publishes every file this notebook wrote to `data/processed/`, `results/metrics/` and `results/figures/` as a new version of the private Kaggle Dataset `cross-subject-brain-encoding-08b-outputs` (created on the first run). The cell first checks that the required outputs exist, so an unfinished run uploads nothing.

Needs Internet on and the Kaggle Secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` attached to this notebook (Add-ons -> Secrets). Without them the cell prints how to set them up and skips.

In [ ]:
from cross_subject_encoding.utils.kaggle_export import export_to_dataset

EXPORT_TO_DATASET = True
exports = [report_path]
if RUN_FULL_FEATURES:
    exports += [C.METRICS_DIR / "08b_selfcheck.csv", C.METRICS_DIR / "08b_headline.json"]
if FEATURES_VERIFIED:
    exports.append(C.DATA_PROCESSED / "eeg1_paper_kpca1000.npz")
print("Outputs:", [str(path) for path in exports])
if EXPORT_TO_DATASET:
    export_to_dataset(slug="cross-subject-brain-encoding-08b-outputs",
                      title="XSUB 08b outputs", files=exports, include_dirs=[],
                      message="verified features" if FEATURES_VERIFIED else "readiness or failed fidelity diagnostics only")